### This file is merely used to test the vector store retrieval capability.

In [6]:
SINGLE = True # Change to True if you want to use single chroma database for all documents
collection_name = "academic_documents" if not SINGLE else "vaa_documents"

from langchain.vectorstores import Chroma
from chromadb.config import Settings
from chromadb import Client, PersistentClient
from langchain_community.embeddings import OllamaEmbeddings
from langchain_community.retrievers import BM25Retriever
from langchain_core.documents import Document

# Now using nomic-embed-text-v2-moe
embedding_function = OllamaEmbeddings(model="bge-m3:567m") # Please OPEN Ollama first!!

query = "What is work integrated education?"
embedding = embedding_function.embed_query(query)

In [7]:
client = Client(Settings())
client = PersistentClient(path="../chroma_db")
collection = client.get_collection(name=collection_name)

vectorStore = Chroma(
    collection_name=collection_name, 
    client=client, 
    embedding_function=embedding_function)

collections = client.get_collection(collection_name)
docs = collections.get(
    include=["documents", "metadatas"],
    limit=collections.count()
)

docs_for_bm25 = [
    Document(page_content=doc_text, metadata=md)
    for doc_text, md in zip(docs["documents"], docs["metadatas"])
]

bm25_retriever = BM25Retriever.from_documents(docs_for_bm25)
print(f"BM25 retriever: initialized over {len(docs_for_bm25)} documents")

BM25 retriever: initialized over 7615 documents


In [8]:
docs = vectorStore.similarity_search(query, k=20)

for doc in docs:
    '''
    if "original_table" in doc.metadata:
        print("[Swapping for Raw Markdown Table]")
        new_result = doc.metadata["original_table"]
    else:
        new_result = doc.page_content
    '''
    
    print("========================================================")
    print(f"Content: {doc.page_content}...")
    print(f"Source: {doc.metadata.get('source')}")
    print(f"Chunk ID: {doc.metadata.get('chunk_id')}\n")

Content: --- Programme booklet: Bachelor of Engineering (Hons) Scheme in Electrical Engineering | BEng(Hons) in EE --- Retrieved from: https://www.polyu.edu.hk/eee/study/information-for-current-students/programme-documents/ ---

Work-Integrated  Education  (WIE)  is  defined  as  a  structured  and  measurable  learning experience  which  takes  place  in  an  organisational  context  relevant  to  a  student's  future profession.  It aims to prepare students for the realities of workplaces, develop students' ability to  learn  in  non-academic  surroundings,  allow  students  to  assess  their  own  strengths  and weaknesses in a realistic working settings and develop students' critical thinking and problem solving capabilities....
Source: BEng_Scheme_EE_46408_2526.pdf
Chunk ID: PolyU_BEng_Scheme_EE_46408_2526.pdf_p46_text_chunk_2131

Content: --- Programme booklet: BEng/BSc (Hons) Scheme in Information and Artificial Intelligence Engineering | BEng/BSc (Hons) in IAIE --- Retrieved fr

In [9]:
bm25_docs = bm25_retriever.get_relevant_documents(query)[:5]

for doc in bm25_docs:
    '''
    if "original_table" in doc.metadata:
        print("[Swapping for Raw Markdown Table]")
        new_result = doc.metadata["original_table"]
    else:
        new_result = doc.page_content
    '''
    
    print("========================================================")
    print(f"Content: {doc.page_content}...")
    print(f"Source: {doc.metadata.get('source')}")
    print(f"Chunk ID: {doc.metadata.get('chunk_id')}\n")

Content: --- Programme booklet: PhD / MPhil in Electrical and Electronic Engineering | PhD/MPhil in EEE --- Retrieved from: https://www.polyu.edu.hk/eee/study/information-for-current-students/programme-documents/ ---

3.4.2 MPhil  and  PhD  theses  shall  consist  of  the  student's  own  work  of  their investigations and be an integrated and coherent piece of work....
Source: PhDMPhil_EEE_46601_2526.pdf
Chunk ID: PolyU_PhDMPhil_EEE_46601_2526.pdf_p14_text_chunk_5390

Content: --- PolyU SAO Website URL: https://www.polyu.edu.hk/sao/counselling-and-wellness-section/programmes-and-activities/polyu-wellmind-go/enquiry/ ---

Enquiry | Student Affairs Office

Email: stud.counselling@polyu.edu.hk
Phone: (852) 2766 6800
What is PolyU WellMind GO
Gift Redemption
Terms & Conditions
Read More
Survey...
Source: https://www.polyu.edu.hk/sao/counselling-and-wellness-section/programmes-and-activities/polyu-wellmind-go/enquiry/
Chunk ID: PolyU_SAO_enquiry_chunk_898

Content: --- Programme booklet: B